# Uber Airtable Unequal:   Number of Customers and Drivers may differ. 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2024, University of Chicago 

---

Here, we extend Uber Air with a new base with unequal number of drivers and customers.

In [3]:
# === SETUP ===
import pulp
import os
from pyairtable import Api
from pprint import pprint

# Portable solver setup, to allow work locally (ARM64 architecture) as well as in JupyterHub
from pulp import COIN_CMD
if os.path.exists('/opt/homebrew/opt/cbc/bin/cbc'):
    solver = COIN_CMD(path='/opt/homebrew/opt/cbc/bin/cbc', msg=0)
else:
    solver = pulp.PULP_CBC_CMD(msg=0)

### Access the UberUnequal base 

#### Do the following:

1. You need to share my Uber database within your AirTable account by, [clicking here](https://airtable.com/invite/l?inviteId=invUa4CeTtIt08Naa&inviteToken=9bc0af8a738f1a910f946658b8d6a46cf493f254ce0f0b216a97d97acf537c48&utm_medium=email&utm_source=product_team&utm_content=transactional-alerts)

1. Click your Account in the upper right corner and select "Developer Hub"

1. Click "..." on far right by your account token and select "Edit"

1. Then click "Add a base" and select "UberUnequal"

In [6]:
AIRTABLE_API_TOKEN = 'patDlJ0vTK7ZXbVpL.63bf7ff0db1d1e8d2f932285d08e17ac3a11bd23395efb4910d6cdfacf91581b'
AIRTABLE_BASE_ID = 'appfjqqXC1MFqsVbt'   # The ID for the UberUnequal Base

CUSTOMER_TABLE_ID = "Customer" # name of the customer table in the Uber base
DRIVER_TABLE_ID = "Driver" # name of the driver table in the Uber base

# create a new client configured with your api token
api = Api(AIRTABLE_API_TOKEN)

# use the client created in the prior cell to get all of the records in the TestTable table
customer_table = api.table(AIRTABLE_BASE_ID, CUSTOMER_TABLE_ID)
driver_table = api.table(AIRTABLE_BASE_ID, DRIVER_TABLE_ID)

### Create our `drivers` and `customers` lists

In [7]:
drivers = []
customers = []
for customer in customer_table.all():
    customers.append(customer['fields']['Num'])
for driver in driver_table.all():
    drivers.append(driver['fields']['Num'])
print(f'Drivers:{drivers}')
print(f'Customers:{customers}')

Drivers:['D32', 'D8712', 'D922', 'D88']
Customers:['C11', 'C934', 'C331']


### Create `customer_addresses` and `driver_locations` 

In [11]:
customer_addresses = {}
for customer in customer_table.all():
    customer_addresses[customer['fields']['Num']] = customer['fields']['Address']
pprint(customer_addresses)

{'C11': '1400 Briarcliff Rd NE, Apt 621, Atlanta, GA 30306',
 'C331': 'Hyatt Centric Midtown Atlanta',
 'C934': 'Taco Mac, Virginia Highlands, Atlanta, GA'}


In [13]:
driver_locations = {}
for driver in driver_table.all():
    driver_locations[driver['fields']['Num']] = [float(driver['fields']['Lat']),float(driver['fields']['Lng'])]
pprint(driver_locations)

{'D32': [33.793711, -84.317408],
 'D8712': [33.77845, -84.400825],
 'D88': [33.77655, -84.320082],
 'D922': [33.775306, -84.396123]}


### Use Google API for distances between each pair of customers and drivers

In [14]:
# import our google_functions, make sure a copy of it is located in this same folder
from google_functions import *

In [15]:
googlemap = create_googlemaps_object()
travel_times = {}
for driver in drivers:
    for customer in customers:
        google_time = duration_in_traffic(googlemap, driver_locations[driver], customer_addresses[customer])
        print(f"Time from {driver} to {customer} is {google_time}")
        travel_times[(driver,customer)]=google_time 
pprint(travel_times)

Time from D32 to C11 is 596
Time from D32 to C934 is 924
Time from D32 to C331 is 1453
Time from D8712 to C11 is 1550
Time from D8712 to C934 is 1053
Time from D8712 to C331 is 499
Time from D922 to C11 is 1491
Time from D922 to C934 is 989
Time from D922 to C331 is 444
Time from D88 to C11 is 572
Time from D88 to C934 is 829
Time from D88 to C331 is 1300
{('D32', 'C11'): 596,
 ('D32', 'C331'): 1453,
 ('D32', 'C934'): 924,
 ('D8712', 'C11'): 1550,
 ('D8712', 'C331'): 499,
 ('D8712', 'C934'): 1053,
 ('D88', 'C11'): 572,
 ('D88', 'C331'): 1300,
 ('D88', 'C934'): 829,
 ('D922', 'C11'): 1491,
 ('D922', 'C331'): 444,
 ('D922', 'C934'): 989}


In [16]:
from pulp import *
model = LpProblem("Uber_Assignment",LpMinimize)

In [21]:
X= {} 
for driver in drivers: 
    for customer in customers:
        X[(driver,customer)] = LpVariable(f"X_{driver}_{customer}", cat='Binary')
print('Flow Variables X:')
pprint(X)

Flow Variables X:
{('D32', 'C11'): X_D32_C11,
 ('D32', 'C331'): X_D32_C331,
 ('D32', 'C934'): X_D32_C934,
 ('D8712', 'C11'): X_D8712_C11,
 ('D8712', 'C331'): X_D8712_C331,
 ('D8712', 'C934'): X_D8712_C934,
 ('D88', 'C11'): X_D88_C11,
 ('D88', 'C331'): X_D88_C331,
 ('D88', 'C934'): X_D88_C934,
 ('D922', 'C11'): X_D922_C11,
 ('D922', 'C331'): X_D922_C331,
 ('D922', 'C934'): X_D922_C934}


In [22]:
# Let's loop through OUR variable names, like 'C1_D2', 
# from those we can access the distance and true PuLP variable objects
obj = ''
for driver in drivers: 
    for customer in customers: 
        obj += travel_times[(driver,customer)] * X[(driver,customer)]
model += obj, "Cost of Customer Driver Assignment"
print(model)

Uber_Assignment:
MINIMIZE
596*X_D32_C11 + 1453*X_D32_C331 + 924*X_D32_C934 + 1550*X_D8712_C11 + 499*X_D8712_C331 + 1053*X_D8712_C934 + 572*X_D88_C11 + 1300*X_D88_C331 + 829*X_D88_C934 + 1491*X_D922_C11 + 444*X_D922_C331 + 989*X_D922_C934 + 0
VARIABLES
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D8712_C11 <= 1 Integer
0 <= X_D8712_C331 <= 1 Integer
0 <= X_D8712_C934 <= 1 Integer
0 <= X_D88_C11 <= 1 Integer
0 <= X_D88_C331 <= 1 Integer
0 <= X_D88_C934 <= 1 Integer
0 <= X_D922_C11 <= 1 Integer
0 <= X_D922_C331 <= 1 Integer
0 <= X_D922_C934 <= 1 Integer



In [23]:
#  Or, of course, let's use Python loops!!
# First the driver constraints
for driver in drivers:
    const = ''
    for customer in customers:
        const += X[(driver,customer)]
    model += const <= 1, f"driver_{driver}"
# Next let's do the customer constraints
for customer in customers:
    const = ''
    for driver in drivers:
        const += X[(driver,customer)]
    model += const <= 1, f"customer_{customer}"

### Add constraint "sum of arcs constraint" to handle unequal number of drivers and customers

In [24]:
sum_arcs = min(len(drivers), len(customers))

arc_sum = ''
for driver in drivers:
    for customer in customers:
        arc_sum += X[(driver, customer)]
model += arc_sum == sum_arcs, "sum_of_arcs"

In [25]:
print(model)

Uber_Assignment:
MINIMIZE
596*X_D32_C11 + 1453*X_D32_C331 + 924*X_D32_C934 + 1550*X_D8712_C11 + 499*X_D8712_C331 + 1053*X_D8712_C934 + 572*X_D88_C11 + 1300*X_D88_C331 + 829*X_D88_C934 + 1491*X_D922_C11 + 444*X_D922_C331 + 989*X_D922_C934 + 0
SUBJECT TO
driver_D32: X_D32_C11 + X_D32_C331 + X_D32_C934 <= 1

driver_D8712: X_D8712_C11 + X_D8712_C331 + X_D8712_C934 <= 1

driver_D922: X_D922_C11 + X_D922_C331 + X_D922_C934 <= 1

driver_D88: X_D88_C11 + X_D88_C331 + X_D88_C934 <= 1

customer_C11: X_D32_C11 + X_D8712_C11 + X_D88_C11 + X_D922_C11 <= 1

customer_C934: X_D32_C934 + X_D8712_C934 + X_D88_C934 + X_D922_C934 <= 1

customer_C331: X_D32_C331 + X_D8712_C331 + X_D88_C331 + X_D922_C331 <= 1

sum_of_arcs: X_D32_C11 + X_D32_C331 + X_D32_C934 + X_D8712_C11 + X_D8712_C331
 + X_D8712_C934 + X_D88_C11 + X_D88_C331 + X_D88_C934 + X_D922_C11
 + X_D922_C331 + X_D922_C934 = 3

VARIABLES
0 <= X_D32_C11 <= 1 Integer
0 <= X_D32_C331 <= 1 Integer
0 <= X_D32_C934 <= 1 Integer
0 <= X_D8712_C11 <= 1 Integ

In [26]:
# Let's solve the model and make sure it's status is good
model.solve(solver)
print("Status:", LpStatus[model.status])

Status: Optimal


In [27]:
# Here is the solution
for v in model.variables():
    print(v.name, "=", v.varValue)

X_D32_C11 = 1.0
X_D32_C331 = 0.0
X_D32_C934 = 0.0
X_D8712_C11 = 0.0
X_D8712_C331 = 0.0
X_D8712_C934 = 0.0
X_D88_C11 = 0.0
X_D88_C331 = 0.0
X_D88_C934 = 1.0
X_D922_C11 = 0.0
X_D922_C331 = 1.0
X_D922_C934 = 0.0


In [29]:
print("Total Objective Function Value is = ", value(model.objective))

Total Objective Function Value is =  1869.0


#### SOLUTION

The following solution may be expected, but it could differ if duration in traffic conditions change significantly:

- X_D32_C11 = 1.0 
- X_D32_C331 = 0.0
- X_D32_C934 = 0.0
- X_D8712_C11 = 0.0
- X_D8712_C331 = 1.0
- X_D8712_C934 = 0.0
- X_D88_C11 = 0.0
- X_D88_C331 = 0.0
- X_D88_C934 = 1.0
- X_D922_C11 = 0.0
- X_D922_C331 = 0.0
- X_D922_C934 = 0.0